# Text3DSAM Inference -- Bilateral Left/Right Positional Bias (Kaggle GPU)

Runs [Text3DSAM](https://github.com/mirthAI/Text3DSAM) (checkpoint: `MagicXin/Text3DSAM`
on Hugging Face) over the **entire** multi-angle rotation sweep in
`samples/text3dsam/manifest_text3dsam_rotated.csv` -- all 3 datasets (`MR_TotalSeg`,
`CT_HaN-Seg`, `CT_TotalSeg_cardiac`), all subjects, all 7 angles, all organs, in one run
-- inference-only, no training.

For every (dataset, subject, rotation_angle_deg) volume, this asks the model to segment
each organ present using ITS OWN prompt (e.g. "Segment the right kidney in this
image."), scores the prediction against the true label (Dice), and records where the
prediction actually landed -- so a later pass can check whether Dice degrades with
rotation angle, and whether "right X" predictions drift toward where "left X" usually
sits.

**Output is ONE CSV with the exact same column schema as the VoxTell rotation-sweep
CSV** (`predictions_voxtell_neurite_oasis_rotation_<PERSON>.csv` from
`voxtell-oasis-rotation-multiangle-kaggle.ipynb`) -- same `gt_mask_rle_b64` /
`pred_mask_rle_b64` / `volume_shape_ras` / `pred_side_axis*` / `gt_side_axis*` columns --
so it can be downloaded and plotted locally with the same plotting code, unmodified.

**This notebook is untested against a live Text3DSAM checkpoint** (built without GPU
access) -- it follows `src/pred.py` in the Text3DSAM repo as closely as possible, but
expect to debug the model-loading/generate() cells interactively on first run.


In [ ]:
import subprocess, sys
from pathlib import Path

# Kaggle's GPU image already ships a working torch+CUDA pair -- do NOT reinstall torch
# here (Text3DSAM's own requirements.txt pins torch==2.7.0+cu128 plus deepspeed,
# causal-conv1d, bitsandbytes, cupy-cuda12x -- all training-only; grep of src/model/**
# confirms NONE of those are actually imported for inference, only: transformers, monai,
# safetensors, calflops, einops, timm). Installing only that inference-minimal set avoids
# the torch/torchvision-mismatch trap the VoxTell notebook had to work around.
_marker = Path('/kaggle/working/.env_setup_done')
if _marker.exists():
    print('environment already installed earlier in this session -- skipping.')
else:
    def _pip(*args):
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

    print('installing inference-only deps (torch left untouched)...')
    _pip('transformers>=4.51,<5', 'monai==1.4.0', 'safetensors', 'calflops', 'einops',
         'timm', 'huggingface_hub', 'nibabel', 'pandas', 'matplotlib')
    _marker.write_text('inference deps installed')
    print('done. (No kernel restart needed -- torch was never touched.)')


## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU visible -- enable GPU under Notebook Settings -> Accelerator."
print("CUDA OK:", torch.cuda.get_device_name(0))


## 2. Setup Working Directories (Kaggle Environment)

In [ ]:
import os
from pathlib import Path

WORKING_DIR = Path('/kaggle/working')
RESULTS_ROOT = WORKING_DIR / 'results'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print('Results will be saved under:', RESULTS_ROOT)


## 3. Clone Text3DSAM (model code)

Public repo, no token needed. `src/pred.py`'s imports only resolve correctly with BOTH
the repo root (for `from src.model...` inside `modeling.py`) AND `src/` itself (for
`from model.modeling import Med3DSeg`, as `pred.py` does) on `sys.path` -- there are no
`__init__.py` files, so this relies on Python 3 namespace packages; adding both
directories explicitly avoids depending on how the cwd happens to be set.


In [ ]:
import subprocess

TEXT3DSAM_DIR = WORKING_DIR / 'Text3DSAM'
if not TEXT3DSAM_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                     'https://github.com/mirthAI/Text3DSAM.git', str(TEXT3DSAM_DIR)],
                    check=True)
else:
    print('already cloned')

import sys
sys.path.insert(0, str(TEXT3DSAM_DIR))          # for "from src.model...." (used inside modeling.py)
sys.path.insert(0, str(TEXT3DSAM_DIR / 'src'))  # for "from model.modeling import Med3DSeg"
print('Text3DSAM at', TEXT3DSAM_DIR)


## 4. Clone your research repo (Text3DSAM samples/manifest)

Uses a Kaggle Secret (`GITHUB_PAT`, added under **Add-ons -> Secrets**) -- this cell never
writes a real token into the notebook file itself. Add the secret before running this cell.


In [ ]:
import shutil
import subprocess

from kaggle_secrets import UserSecretsClient

_gh_token = UserSecretsClient().get_secret("GITHUB_PAT")
assert _gh_token, "Add a Kaggle Secret named GITHUB_PAT (Add-ons -> Secrets) before running this cell."

_clone_url = f'https://{_gh_token}@github.com/fatemafaria142/positional-bias-medical-vlms.git'
_dest = '/kaggle/working/research_repo'
shutil.rmtree(_dest, ignore_errors=True)

_result = subprocess.run(
    ['git', 'clone', '--branch', 'faria/text3dsam-multi-dataset-experiments', '--depth', '1',
     _clone_url, _dest],
    capture_output=True, text=True,
)
_stderr_scrubbed = _result.stderr.replace(_gh_token, '***')
del _gh_token, _clone_url

if _result.returncode != 0:
    print('git clone FAILED:')
    print(_stderr_scrubbed)
else:
    print('cloned OK')

RESEARCH_REPO = Path('/kaggle/working/research_repo')
DATA_ROOT = RESEARCH_REPO / 'samples' / 'text3dsam'
print('datasets present:', [p.name for p in DATA_ROOT.iterdir() if p.is_dir()])


## 5. Download and load the Text3DSAM model

Single published checkpoint: `MagicXin/Text3DSAM` (safetensors, ~119 MB + a small
`nlpie/tiny-clinicalbert` text encoder pulled in automatically via its `text_model`
config field). Loaded in float32 regardless of the checkpoint's stored dtype
(`config.json` says `bfloat16`) since this is inference-only and not every Kaggle GPU
(e.g. an older T4) handles bf16 well.


In [ ]:
import torch
from transformers import BertTokenizer
from model.modeling import Med3DSeg  # registers Med3DSegConfig/Med3DSeg with Auto* as a side effect

MODEL_REPO = 'MagicXin/Text3DSAM'
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

tokenizer = BertTokenizer.from_pretrained(MODEL_REPO)
model = Med3DSeg.from_pretrained(MODEL_REPO, torch_dtype=torch.float32).to(device).eval()

IMAGE_SIZE = tuple(model.config.image_size)  # (128, 256, 256) per the published config
print('Text3DSAM loaded on', device, '-- image_size:', IMAGE_SIZE)


## 6. Helpers

Dice / RLE mask encoding / panel figure, matching `scripts/common3d.py`'s
`mask_ground_truth_3d` policy (side derived from geometric center -- fine here since we
only use it on the PREDICTED mask for a rough "where did it actually land" read; the true
`gt_side_axis*` columns already carry the authoritative organ-name-derived side, per the
manifest fix, and are copied straight from the manifest below, never recomputed).


In [ ]:
import base64
import csv
import time
import zlib

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import monai.transforms as mtf
import nibabel as nib
import numpy as np
import torch.nn.functional as F

_OPPOSITE = {"R": "L", "L": "R", "A": "P", "P": "A", "S": "I", "I": "S"}

def mask_ground_truth_3d(mask: np.ndarray, axcodes: tuple):
    coords = np.nonzero(mask)
    if len(coords[0]) == 0:
        return None
    centroid = tuple(float(c.mean()) for c in coords)
    bbox = tuple(int(c.min()) for c in coords) + tuple(int(c.max()) for c in coords)
    sides = [code if val > mask.shape[axis] / 2 else _OPPOSITE[code]
             for axis, (code, val) in enumerate(zip(axcodes, centroid))]
    return centroid, bbox, tuple(sides)

def dice_score(pred: np.ndarray, gt: np.ndarray) -> float:
    inter = np.logical_and(pred, gt).sum()
    denom = pred.sum() + gt.sum()
    return 1.0 if denom == 0 else float(2.0 * inter / denom)

def struct_slug(name: str) -> str:
    return name.strip().lower().replace(' ', '_').replace('/', '-')

def window(img_slice):
    lo, hi = np.percentile(img_slice, [1, 99])
    if hi <= lo:
        return img_slice
    return np.clip((img_slice - lo) / (hi - lo), 0, 1)

def encode_mask_rle(mask: np.ndarray) -> str:
    flat = np.asarray(mask, dtype=bool).ravel(order='C')
    change_idx = np.flatnonzero(flat[1:] != flat[:-1]) + 1
    boundaries = np.concatenate(([0], change_idx, [flat.size]))
    runs = np.diff(boundaries).astype(np.uint32)
    if flat.size and flat[0]:
        runs = np.concatenate(([0], runs)).astype(np.uint32)
    raw = zlib.compress(runs.tobytes(), level=9)
    return base64.b64encode(raw).decode('ascii')

def decode_mask_rle(encoded: str, shape) -> np.ndarray:
    runs = np.frombuffer(zlib.decompress(base64.b64decode(encoded)), dtype=np.uint32)
    flat = np.zeros(int(np.prod(shape)), dtype=bool)
    idx = 0
    value = False
    for run_len in runs:
        if value and run_len:
            flat[idx:idx + run_len] = True
        idx += int(run_len)
        value = not value
    return flat.reshape(shape, order='C')

GT_COLOR = (0.13, 0.70, 0.30)
PRED_COLOR = (0.85, 0.15, 0.15)
plt.rcParams.update({
    'axes.titlesize': 12, 'axes.titleweight': 'medium',
    'figure.titlesize': 13, 'legend.fontsize': 10,
    'savefig.facecolor': 'white',
})

def save_panel_figure(out_path, prompt_text, dice, img_slice, gt_slice, pred_slice):
    h, w = img_slice.shape
    fig, axes = plt.subplots(1, 3, figsize=(9.6, 9.6 * h / w / 3 + 0.9),
                              gridspec_kw={'wspace': 0.02})
    for ax, label in zip(axes, ('Original', 'Ground Truth', 'Prediction')):
        ax.imshow(img_slice, cmap='gray', origin='lower')
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.set_xlabel(label, fontsize=11, labelpad=6)
    gt_overlay = np.zeros((*gt_slice.shape, 4)); gt_overlay[gt_slice] = (*GT_COLOR, 0.5)
    axes[1].imshow(gt_overlay, origin='lower')
    pred_overlay = np.zeros((*pred_slice.shape, 4)); pred_overlay[pred_slice] = (*PRED_COLOR, 0.5)
    axes[2].imshow(pred_overlay, origin='lower')
    legend_handles = [
        plt.Line2D([0], [0], marker='s', color='none', markerfacecolor=GT_COLOR, markersize=12, label='Ground Truth'),
        plt.Line2D([0], [0], marker='s', color='none', markerfacecolor=PRED_COLOR, markersize=12, label='Prediction'),
    ]
    fig.legend(handles=legend_handles, loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.02), frameon=False)
    fig.suptitle(f'"{prompt_text}"', fontsize=13, fontweight='bold', y=1.06)
    ax_title_color = PRED_COLOR if dice < 0.3 else '#1a1a1a'
    axes[1].set_title(f'Dice = {dice:.3f}', fontsize=12, fontweight='bold', color=ax_title_color, pad=8)
    fig.subplots_adjust(left=0.01, right=0.99, top=0.90, bottom=0.16, wspace=0.02)
    fig.savefig(out_path, dpi=200, bbox_inches='tight', pad_inches=0.15)
    plt.close(fig)

print('helpers ready')


## 7. Model call -- image encoded ONCE per volume, reused across organs

`Med3DSeg.generate()` (in the published repo) recomputes the image encoding on every
call, which is wasteful when scoring several organs against the same volume. This
inlines the same two lines `generate()` runs, but caches `image_features` per volume so
each organ only re-runs the (cheap) text encoder + segmentation decoder.


In [ ]:
@torch.no_grad()
def encode_image(image_tensor):
    """image_tensor: (1, 1, D, H, W) float32 on `device`, already resized to IMAGE_SIZE."""
    return model.image_encoder(image_tensor)

@torch.no_grad()
def predict_one_organ(image_features, prompt_text, origin_shape):
    """Returns a binary (D0, H0, W0) numpy mask at the volume's ORIGINAL shape,
    thresholded at 0.5 -- same as pred.py's instance_label==1 branch, appropriate here
    since every manifest row already names exactly one organ."""
    tokens = tokenizer(prompt_text, padding='max_length', truncation=True, max_length=512,
                        return_tensors='pt').to(device)
    text_feature = model.text_encoder(
        input_ids=tokens['input_ids'], attention_mask=tokens['attention_mask']
    )['last_hidden_state'][:, 0]
    text_embedding = model.text_projector(text_feature)
    logits = model.seg_decoder(image_features, text_embedding)
    logits = F.interpolate(logits, size=origin_shape, mode='trilinear', align_corners=False)
    prob = torch.sigmoid(logits)
    return (prob > 0.5).squeeze(0).squeeze(0).cpu().numpy().astype(bool)

pre_resize = mtf.Compose([
    mtf.EnsureChannelFirst(channel_dim='no_channel'),
    mtf.Resize(spatial_size=IMAGE_SIZE, mode='trilinear'),
    mtf.ToTensor(dtype=torch.float32),
])

def load_and_prepare_image(image_path):
    """Loads the nii.gz in its stored array order (no reorientation -- our volumes were
    built directly from CVPR-BiomedSegFM's own npz arrays with a matching custom affine,
    so nibabel's native load order already equals what the model was trained on)."""
    arr = np.asarray(nib.load(str(image_path)).dataobj).astype(np.float32)
    origin_shape = arr.shape
    resized = pre_resize(arr).unsqueeze(0).to(device)  # (1, 1, *IMAGE_SIZE)
    return resized, origin_shape

print('model-call helpers ready')


## 8. Dataset config -- the WHOLE manifest, all 3 datasets

`PERSON` only exists so the output CSV's schema matches VoxTell's exactly (that column
meant "which team member's subjects" there; this repo isn't split by annotator, so it's
just a fixed label here) -- change it if you want a different value in that column.


In [ ]:
PERSON = 'faria'  # only kept for exact column-schema parity with the VoxTell CSV

ANGLES_DEG = [0, 30, 60, 90, 120, 150, 180]
N_SUBJECTS_PER_DATASET = None  # None = every subject in every dataset; set an int for a quick test

MANIFEST_PATH = DATA_ROOT / 'manifest_text3dsam_rotated.csv'

def load_manifest_grouped(manifest_path):
    """Group ALL rows (every dataset) by (dataset, subject_id, rotation_angle_deg) --
    each triple is a physically distinct rotated volume file, and every row sharing that
    triple shares the same image (just a different organ prompt)."""
    groups = {}
    with open(manifest_path, newline='') as f:
        for row in csv.DictReader(f):
            key = (row['dataset'], row['subject_id'], int(row['rotation_angle_deg']))
            groups.setdefault(key, []).append(row)
    return groups

grouped = load_manifest_grouped(MANIFEST_PATH)

if N_SUBJECTS_PER_DATASET is not None:
    subjects_by_dataset = {}
    for (ds, sid, _angle) in grouped:
        subjects_by_dataset.setdefault(ds, set()).add(sid)
    kept_subjects = {ds: sorted(sids)[:N_SUBJECTS_PER_DATASET] for ds, sids in subjects_by_dataset.items()}
    all_groups = {(ds, sid, angle): rows for (ds, sid, angle), rows in grouped.items()
                  if sid in kept_subjects.get(ds, [])}
else:
    all_groups = grouped

n_rows_total = sum(len(rows) for rows in all_groups.values())
datasets_present = sorted({ds for ds, _sid, _angle in all_groups})
for ds in datasets_present:
    ds_groups = {k: v for k, v in all_groups.items() if k[0] == ds}
    ds_subjects = sorted({sid for _ds, sid, _angle in ds_groups})
    print(f'{ds}: {len(ds_subjects)} subject(s), {len(ds_groups)} (subject, angle) groups, '
          f'{sum(len(v) for v in ds_groups.values())} organ-prompt rows')
print(f'\nTOTAL: {len(all_groups)} (dataset, subject, angle) groups, {n_rows_total} organ-prompt rows')


## 9. Smoke test -- ONE (subject, angle) group, all its organs

Time this before committing to the full run.


In [ ]:
(smoke_dataset, smoke_subject_id, smoke_angle), smoke_rows = next(iter(all_groups.items()))
smoke_organs = [r['organ'] for r in smoke_rows]
smoke_image_path = DATA_ROOT / smoke_dataset / smoke_rows[0]['image_path']
print(f'smoke test: {smoke_dataset}/{smoke_subject_id} @ {smoke_angle} deg -- organs:', smoke_organs)

t0 = time.time()
smoke_img_tensor, smoke_origin_shape = load_and_prepare_image(smoke_image_path)
smoke_features = encode_image(smoke_img_tensor)
smoke_masks = [predict_one_organ(smoke_features, r['prompt_text'], smoke_origin_shape) for r in smoke_rows]
smoke_elapsed = time.time() - t0

print(f'smoke test elapsed: {smoke_elapsed:.1f}s for {len(smoke_organs)} organ(s) '
      f'(1 shared image encoding + {len(smoke_organs)} text/decoder passes)')
print('original volume shape:', smoke_origin_shape, '-- resized to:', IMAGE_SIZE)
print('predicted mask voxel counts:', [int(m.sum()) for m in smoke_masks])


## 10. Runtime estimate for the WHOLE manifest

In [ ]:
per_group_seconds = smoke_elapsed
n_groups = len(all_groups)

print(f'Measured: {per_group_seconds:.1f}s for 1 group on this GPU.')
print(f'Whole manifest ({n_groups} groups, {n_rows_total} rows, all 3 datasets): '
      f'~{per_group_seconds * n_groups / 60:.1f} minutes estimated.')


## 11. Full run -- one shared image encoding per (dataset, subject, angle), one decoder pass per organ

**How results are saved -- ONE combined CSV, same schema as the VoxTell rotation-sweep CSV:**

- **CSV** -- `results/predictions_text3dsam_{PERSON}.csv`, one row per
  (dataset, subject, angle, organ), across ALL 3 datasets. Column names match
  `predictions_voxtell_neurite_oasis_rotation_<PERSON>.csv` exactly: `person`, `dataset`,
  `subject_id`, `rotation_angle_deg`, `split`, `organ`, `label_id`, `prompt_text`,
  `image_path`, `label_path`, `volume_shape_ras`, `n_pred_voxels`, `dice`,
  `inference_seconds`, `gt_mask_rle_b64`, `pred_mask_rle_b64`, `pred_centroid_vox_*`,
  `pred_bbox_vox_*`, `pred_side_axis*`, `gt_centroid_vox_*`, `gt_side_axis*` -- so it
  drops into the same local plotting code unmodified. (`volume_shape_ras` is named to
  match that schema, but Text3DSAM's volumes are NOT reoriented to RAS the way VoxTell's
  loader does -- it's each volume's own native array shape/order; the true axis meaning
  is in the extra `axcodes` column this notebook also writes.)
- **PNGs** -- `results/results_text3dsam/<dataset>/<subject>_rot<angle>/<organ>.png`,
  a 3-panel (original / ground truth / prediction) figure per case.
- **Masks** -- `results/masks_text3dsam/<dataset>/<organ>/<subject>_rot<angle>.npz`.

Resumable: re-running skips (dataset, subject, angle, organ) rows already present in the CSV.


In [ ]:
def run_text3dsam_full_manifest():
    fig_root = RESULTS_ROOT / 'results_text3dsam'
    fig_root.mkdir(parents=True, exist_ok=True)
    masks_root = RESULTS_ROOT / 'masks_text3dsam'
    masks_root.mkdir(parents=True, exist_ok=True)
    csv_path = RESULTS_ROOT / f'predictions_text3dsam_{PERSON}.csv'

    # Exact column names from predictions_voxtell_neurite_oasis_rotation_<PERSON>.csv,
    # plus one extra (axcodes) appended at the end -- pandas/CSV readers keyed by column
    # name ignore extra columns, so this stays a drop-in read for existing plotting code.
    fieldnames = ['person', 'dataset', 'subject_id', 'rotation_angle_deg', 'split', 'organ',
                  'label_id', 'prompt_text', 'image_path', 'label_path', 'volume_shape_ras',
                  'n_pred_voxels', 'dice', 'inference_seconds', 'gt_mask_rle_b64', 'pred_mask_rle_b64',
                  'pred_centroid_vox_0', 'pred_centroid_vox_1', 'pred_centroid_vox_2',
                  'pred_bbox_vox_0min', 'pred_bbox_vox_1min', 'pred_bbox_vox_2min',
                  'pred_bbox_vox_0max', 'pred_bbox_vox_1max', 'pred_bbox_vox_2max',
                  'pred_side_axis0', 'pred_side_axis1', 'pred_side_axis2',
                  'gt_centroid_vox_0', 'gt_centroid_vox_1', 'gt_centroid_vox_2',
                  'gt_side_axis0', 'gt_side_axis1', 'gt_side_axis2', 'axcodes']

    already_done = set()
    write_header = not csv_path.exists()
    if csv_path.exists():
        with open(csv_path, newline='') as f:
            for row in csv.DictReader(f):
                already_done.add((row['dataset'], row['subject_id'], int(row['rotation_angle_deg']), row['organ']))
        print(f'Resuming: found {len(already_done)} already completed cases in {csv_path.name}')

    with open(csv_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
            f.flush()

        for gi, ((dataset_name, subject_id, angle), rows) in enumerate(all_groups.items(), 1):
            pending = [r for r in rows if (dataset_name, subject_id, angle, r['organ']) not in already_done]
            if not pending:
                print(f'[{gi}/{len(all_groups)}] {dataset_name}/{subject_id} @ {angle} deg: already completed, skipping')
                continue

            # The manifest's image_path/label_path are relative to each dataset's OWN
            # folder (e.g. "volumes_rot/CT_HaN-Seg_case_01_rot000_image.nii.gz"), NOT
            # already dataset-prefixed -- the real file lives at
            # samples/text3dsam/<dataset>/volumes_rot/..., so `dataset_name` must be
            # inserted here explicitly (this was missing before and caused
            # FileNotFoundError: .../samples/text3dsam/volumes_rot/... on Kaggle).
            image_path = DATA_ROOT / dataset_name / rows[0]['image_path']
            label_path = DATA_ROOT / dataset_name / rows[0]['label_path']
            axcodes = tuple(rows[0]['axcodes'])

            print(f'[{gi}/{len(all_groups)}] {dataset_name}/{subject_id} @ {angle} deg: '
                  f'encoding image + predicting {len(pending)} organ(s)...')
            t0 = time.time()
            img_tensor, origin_shape = load_and_prepare_image(image_path)
            image_features = encode_image(img_tensor)
            label_vol = np.asarray(nib.load(str(label_path)).dataobj).astype(np.int32)

            for row0 in pending:
                organ = row0['organ']
                label_id = int(row0['label_id'])

                t_organ = time.time()
                pred_mask = predict_one_organ(image_features, row0['prompt_text'], origin_shape)
                inference_seconds = time.time() - t_organ

                gt_mask = label_vol == label_id
                dice = dice_score(pred_mask, gt_mask)
                n_voxels = int(pred_mask.sum())
                gt = mask_ground_truth_3d(pred_mask, axcodes) if n_voxels > 0 else None

                out_row = {
                    'person': PERSON, 'dataset': dataset_name, 'subject_id': subject_id,
                    'rotation_angle_deg': angle, 'split': row0['split'], 'organ': organ,
                    'label_id': label_id, 'prompt_text': row0['prompt_text'],
                    # Stored resolvable from the REPO ROOT (matching how the VoxTell
                    # CSVs store image_path/label_path) -- the manifest's own
                    # image_path/label_path are only "volumes_rot/...", so the dataset
                    # folder is prefixed here explicitly (NOT already dataset-prefixed).
                    'image_path': f"samples/text3dsam/{dataset_name}/{row0['image_path']}",
                    'label_path': f"samples/text3dsam/{dataset_name}/{row0['label_path']}",
                    'volume_shape_ras': 'x'.join(str(s) for s in origin_shape),
                    'n_pred_voxels': n_voxels, 'dice': round(dice, 4),
                    'inference_seconds': round(inference_seconds, 2),
                    'gt_mask_rle_b64': encode_mask_rle(gt_mask),
                    'pred_mask_rle_b64': encode_mask_rle(pred_mask),
                    'pred_centroid_vox_0': None, 'pred_centroid_vox_1': None, 'pred_centroid_vox_2': None,
                    'pred_bbox_vox_0min': None, 'pred_bbox_vox_1min': None, 'pred_bbox_vox_2min': None,
                    'pred_bbox_vox_0max': None, 'pred_bbox_vox_1max': None, 'pred_bbox_vox_2max': None,
                    'pred_side_axis0': None, 'pred_side_axis1': None, 'pred_side_axis2': None,
                    'gt_centroid_vox_0': row0['gt_centroid_vox_0'], 'gt_centroid_vox_1': row0['gt_centroid_vox_1'],
                    'gt_centroid_vox_2': row0['gt_centroid_vox_2'],
                    'gt_side_axis0': row0['gt_side_axis0'], 'gt_side_axis1': row0['gt_side_axis1'],
                    'gt_side_axis2': row0['gt_side_axis2'], 'axcodes': row0['axcodes'],
                }
                if gt is not None:
                    (cx, cy, cz), bbox, (s0, s1, s2) = gt
                    out_row.update({
                        'pred_centroid_vox_0': cx, 'pred_centroid_vox_1': cy, 'pred_centroid_vox_2': cz,
                        'pred_bbox_vox_0min': bbox[0], 'pred_bbox_vox_1min': bbox[1], 'pred_bbox_vox_2min': bbox[2],
                        'pred_bbox_vox_0max': bbox[3], 'pred_bbox_vox_1max': bbox[4], 'pred_bbox_vox_2max': bbox[5],
                        'pred_side_axis0': s0, 'pred_side_axis1': s1, 'pred_side_axis2': s2,
                    })
                writer.writerow(out_row)
                f.flush()
                already_done.add((dataset_name, subject_id, angle, organ))
                print(f'    {organ}: dice={dice:.3f}, pred_voxels={n_voxels} ({inference_seconds:.2f}s)')

                organ_mask_dir = masks_root / dataset_name / struct_slug(organ)
                organ_mask_dir.mkdir(parents=True, exist_ok=True)
                np.savez_compressed(organ_mask_dir / f'{subject_id}_rot{angle:03d}.npz', segs=pred_mask)

                slice_axis = int(np.argmin(origin_shape))
                gt_coords = np.nonzero(gt_mask)
                slice_idx = int(round(gt_coords[slice_axis].mean())) if len(gt_coords[0]) else origin_shape[slice_axis] // 2
                def take_slice(vol, ax=slice_axis, idx=slice_idx):
                    sl = [slice(None)] * 3; sl[ax] = idx
                    return vol[tuple(sl)].T

                img_arr = np.asarray(nib.load(str(image_path)).dataobj).astype(np.float32)
                subj_fig_dir = fig_root / dataset_name / f'{subject_id}_rot{angle:03d}'
                subj_fig_dir.mkdir(parents=True, exist_ok=True)
                save_panel_figure(
                    subj_fig_dir / f'{struct_slug(organ)}.png',
                    row0['prompt_text'], dice,
                    window(take_slice(img_arr)), take_slice(gt_mask), take_slice(pred_mask),
                )

            elapsed_total = time.time() - t0
            print(f'  group done in {elapsed_total:.1f}s')

    print(f'all done -> {csv_path}, {fig_root}')

run_text3dsam_full_manifest()


## 12. Positional-bias analysis -- Dice vs. rotation angle

Does Dice drop as the organ's on-screen position changes (angle increases), even though
the prompt and the true anatomical side never change? A flat line means the model
follows the prompt/anatomy; a downward slope means predictions drift with on-screen
position instead.


In [ ]:
import pandas as pd

pred_df = pd.read_csv(RESULTS_ROOT / f'predictions_text3dsam_{PERSON}.csv')

angle_summary = pred_df.groupby('rotation_angle_deg')['dice'].agg(['mean', 'std', 'count']).reset_index()
print('All 3 datasets combined:')
print(angle_summary)

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(angle_summary['rotation_angle_deg'], angle_summary['mean'], yerr=angle_summary['std'],
            marker='o', capsize=4, color='#2563eb')
ax.set_xlabel('Rotation angle (degrees)')
ax.set_ylabel('Mean Dice (all datasets, organs, subjects)')
ax.set_title('Text3DSAM Dice vs. rotation angle -- all 3 datasets')
ax.set_xticks(ANGLES_DEG)
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
fig.tight_layout()
dice_vs_angle_path = RESULTS_ROOT / 'dice_vs_angle_all_datasets.png'
fig.savefig(dice_vs_angle_path, dpi=150)
plt.close(fig)
print('wrote', dice_vs_angle_path)

# Same curve, split out per dataset -- three datasets can behave very differently.
fig, ax = plt.subplots(figsize=(8, 5))
for ds, grp in pred_df.groupby('dataset'):
    ds_summary = grp.groupby('rotation_angle_deg')['dice'].mean()
    ax.plot(ds_summary.index, ds_summary.values, marker='o', label=ds)
ax.set_xlabel('Rotation angle (degrees)')
ax.set_ylabel('Mean Dice')
ax.set_title('Text3DSAM Dice vs. rotation angle, by dataset')
ax.set_xticks(ANGLES_DEG)
ax.set_ylim(0, 1)
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
dice_vs_angle_by_dataset_path = RESULTS_ROOT / 'dice_vs_angle_by_dataset.png'
fig.savefig(dice_vs_angle_by_dataset_path, dpi=150)
plt.close(fig)
print('wrote', dice_vs_angle_by_dataset_path)

organ_angle = pred_df.groupby(['dataset', 'organ', 'rotation_angle_deg'])['dice'].mean().unstack('rotation_angle_deg')
organ_angle_path = RESULTS_ROOT / 'dice_by_dataset_organ_and_angle.csv'
organ_angle.to_csv(organ_angle_path)
print('wrote', organ_angle_path)
organ_angle.round(3)


## 13. Left/right confusion check

Specifically: when asked for the "right X", does the prediction's own geometric side
(`pred_side_axis2`) actually land on `R`? A model with no positional bias should match
`gt_side_axis2` here almost always, regardless of rotation angle.


In [ ]:
valid = pred_df[pred_df['n_pred_voxels'] > 0].copy()
valid['side_match'] = valid['pred_side_axis2'] == valid['gt_side_axis2']

print('Overall L/R side match rate (predictions with any voxels):',
      f"{valid['side_match'].mean():.1%}", f"({len(valid)} of {len(pred_df)} rows had a non-empty prediction)")
print()
print('Side match rate by rotation angle:')
print(valid.groupby('rotation_angle_deg')['side_match'].mean().round(3))
print()
print('Side match rate by organ:')
print(valid.groupby('organ')['side_match'].mean().round(3))


## 14. Summary & Output Archiving

In [ ]:
import shutil

csv_path = RESULTS_ROOT / f'predictions_text3dsam_{PERSON}.csv'
if csv_path.exists():
    df = pd.read_csv(csv_path)
    print(f'=== Text3DSAM -- ALL 3 DATASETS Summary ===')
    print(f'Total evaluated cases: {len(df)}')
    print(f"Mean Dice: {df['dice'].mean():.4f}")
    print()
    print('Mean Dice by dataset:')
    print(df.groupby('dataset')['dice'].mean().round(4))
    print()
    print('Mean Dice by rotation angle:')
    print(df.groupby('rotation_angle_deg')['dice'].mean().round(4))
    print()
    print('Mean Dice by organ:')
    print(df.groupby('organ')['dice'].mean().sort_values(ascending=False).round(4))

zip_path = WORKING_DIR / 'text3dsam_all_datasets_results'
shutil.make_archive(str(zip_path), 'zip', str(RESULTS_ROOT))
print(f"\nArchive created at {zip_path}.zip ({os.path.getsize(str(zip_path) + '.zip') / 1e6:.1f} MB)")
print("This zip file will appear directly in Kaggle's Output panel.")


## 15. Downloading the results

**You only need `predictions_text3dsam_{PERSON}.csv`** (e.g.
`predictions_text3dsam_faria.csv`) -- one CSV, all 3 datasets, all subjects, all 7
angles, all organs. Every row already carries `gt_mask_rle_b64`/`pred_mask_rle_b64` plus
`volume_shape_ras` to decode them, and the column names match the VoxTell rotation-sweep
CSV exactly, so your existing local plotting code should read it without changes. Find
it in the Kaggle **Output** tab after a **Save Version** run finishes (not just an
interactive Run All).

The zip from the previous cell has the CSV again plus the per-case PNGs and raw `.npz`
prediction masks (organized as `<dataset>/<subject>_rot<angle>/<organ>.png` and
`<dataset>/<organ>/<subject>_rot<angle>.npz`) -- only grab it if you want the
pre-rendered figures too.
